# Canadian housing + financial portfolio — run and checks

This notebook runs the **primary** `portfolio_olg` path and verifies:

1. Joint Canada return chain looks sensible
2. Life-cycle DP **solves** and simulates
3. Lifecycle cohort means and major quantiles (net worth, wealth, c, y)
4. Simulation modes: cohort / panel × drawn / provided initials
5. `DecisionSimulation` callable (solve → both sims → placeholder stats)
6. OLG inequality panel runs (low vs high financial vol)
7. Wealth-dependent financial volatility simulation layer runs

**Model equations:** see [`docs/MODEL.md`](../docs/MODEL.md) (full write-up). A short summary is in the next cell.

Open from `portfolio_olg/notebooks/` (or set the working directory to `portfolio_olg`).

**Runtime:** first Numba compile + solve can take ~15–40s per DP cell.


## Model equations (summary)

Full equations, timing, and code map: **[`../docs/MODEL.md`](../docs/MODEL.md)** (also linked from [`portfolio_olg/README.md`](../README.md)).

### Age grid and period length

The life cycle **always** spans chronological ages $[\texttt{age\_min},\texttt{age\_max}]$ (defaults **25** to **80**), discretized into `T` equally spaced periods:

$$
\mathrm{age}(t)=\texttt{age\_min}+t\cdot\Delta,
\qquad
\Delta=\frac{\texttt{age\_max}-\texttt{age\_min}}{T-1}=\texttt{period\_years}.
$$

So $t\to t+1$ advances age by $\Delta$ years; $\mathrm{age}(0)=25$ and $\mathrm{age}(T-1)=80$. See `model.ages` / `model.period_years` (`src/age_grid.py`). Annual $\beta$, $\delta$, income AR(1), and return placeholders are compounded to frequency $\Delta$.

**Utility**

$$
u(c,h)=\frac{\bigl(c^{\alpha}(h+\underline{h})^{1-\alpha}\bigr)^{1-\rho}-1}{1-\rho}
$$

**Within period:** enter with cash-on-hand $m$, housing $h$, income state $z$, joint return state $s$. Choose keep or adjust:

- **Keep:** $a=m-c\ge 0$, $h^{\mathrm{end}}=h$
- **Adjust:** $x=m+(1-\tau)h$, then $c+a+h^{\mathrm{new}}=x$, $h^{\mathrm{end}}=h^{\mathrm{new}}$

**Laws of motion**

$$
m'=R^{f\prime}\,a+z',\qquad
h'=R^{h\prime}\,(1-\delta)\,h^{\mathrm{end}}
$$

**Bellman:** $V_t=\max\{V^{\mathrm{keep}}_t,V^{\mathrm{adj}}_t\}$ with continuation $\beta\,\mathbb{E}[V_{t+1}(m',h',z',s')]$. Terminal period maximizes period utility only.

**Returns:** bivariate AR(1) in demeaned log returns $(\varepsilon^f,\varepsilon^h)$, annual placeholders compounded to $\Delta$, then discretized.

**Optional sim layer:** $\sigma_f(W)=\sigma_f^{\mathrm{base}}(1+\kappa W/\bar W)$.


## 0. Setup

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

print("ROOT:", ROOT)
print("Python path head:", sys.path[0])

In [ ]:
from src.housing_finance import (
    DecisionSimulation,
    HousingFinanceModel,
    HousingFinanceParams,
    InitialDistribution,
    InitialSample,
)
from src.inequality import gini, summarize_path, wealth_inequality_path
from src.lifecycle_profiles import (
    DEFAULT_VARIABLES,
    PANEL_VARIABLES,
    plot_lifecycle_profiles,
    profiles_from_sim,
    cohort_lifecycle_profiles,
)
from src.returns_ca import CanadaReturnParams, discretize_canada_returns, scale_financial_vol

import time

checks = {}  # name -> bool
print("Imports OK")


## 1. Check Canada joint return chain

Expect: `mean_Rf ≈ 1.06`, `mean_Rh ≈ 1.04`, positive vols, finite correlation.

In [ ]:
ret_params = CanadaReturnParams(
    n_f=3,
    n_h=3,
    sigma_f=0.16,
    sigma_h=0.10,
    corr=0.35,
    Rf_bar=1.06,
    Rh_bar=1.04,
)
chain = discretize_canada_returns(ret_params)
summary = chain.summary()
display(pd.Series(summary))

ok_returns = (
    np.isfinite(summary["mean_Rf"])
    and np.isfinite(summary["mean_Rh"])
    and abs(summary["mean_Rf"] - ret_params.Rf_bar) < 1e-6
    and abs(summary["mean_Rh"] - ret_params.Rh_bar) < 1e-6
    and summary["std_Rf"] > 0
    and summary["std_Rh"] > 0
    and np.isfinite(summary["corr_Rf_Rh"])
)
checks["returns_ca"] = bool(ok_returns)
print("PASS" if ok_returns else "FAIL", "— Canada return chain")

## 2. Solve + simulate life-cycle DP (main solve check)

Success: finite `V[0]`, positive simulation means, figure saved.

In [ ]:
par = HousingFinanceParams(
    T=20,
    age_min=25.0,
    age_max=80.0,
    Nm=14,
    Nh=10,
    Nz=3,
    n_c=10,
    simN=1000,
    returns=ret_params,
    m_max=6.0,
    h_max=4.0,
)
model = HousingFinanceModel(par)

print(f"Age grid ({par.age_min:.0f}-{par.age_max:.0f}): {model.ages}")
print(f"period_years Delta = {model.period_years:.4f}")

print("Solving...")
model.solve()
v0 = float(np.mean(model.sol.V[0]))
print(f"V[0] mean = {v0:.4f}")

print("Simulating...")
model.simulate()

sim_stats = {
    "mean_c": float(model.sim.c.mean()),
    "mean_h": float(model.sim.h.mean()),
    "mean_m": float(model.sim.m.mean()),
    "mean_net_worth": float(model.sim.net_worth.mean()),
    "adj_rate": float(model.sim.adj.mean()),
    "period_years": float(model.period_years),
}
display(pd.Series(sim_stats))

ok_solve = (
    np.isfinite(v0)
    and np.isfinite(sim_stats["mean_c"])
    and sim_stats["mean_c"] > 0
    and sim_stats["mean_net_worth"] > 0
    and 0.0 <= sim_stats["adj_rate"] <= 1.0
    and abs(model.ages[0] - par.age_min) < 1e-12
    and abs(model.ages[-1] - par.age_max) < 1e-12
)
checks["housing_finance_solve_sim"] = bool(ok_solve)
print("PASS" if ok_solve else "FAIL", "— life-cycle solve + simulate")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ages = model.ages
ax.plot(ages, model.sim.m.mean(axis=1), label="liquid m")
ax.plot(ages, model.sim.h.mean(axis=1), label="housing h")
ax.plot(ages, model.sim.c.mean(axis=1), label="consumption c")
ax.plot(ages, model.sim.net_worth.mean(axis=1), label="net worth")
ax.set_xlabel("Age")
ax.set_title(
    f"Housing-finance baseline (ages {par.age_min:.0f}–{par.age_max:.0f}, "
    f"Δ={model.period_years:.2f}y)"
)
ax.legend()
fig.tight_layout()
fig_path = OUT / "housing_finance_baseline.png"
fig.savefig(fig_path, dpi=120)
plt.show()
print("Wrote", fig_path)


## 3. Lifecycle cohort: means and major quantiles

Simulate one birth cohort over periods $t=0,\ldots,T-1$ with chronological ages from **age_min=25** to **age_max=80**. Report, by **age**:

| Series | Meaning |
|--------|---------|
| `net_worth` | $m + h$ at the start of the period |
| `m` | Liquid cash-on-hand |
| `h` | Housing stock |
| `a` | End-of-period liquid saving |
| `c` | Consumption |
| `y` | Labor income (current productivity state) |

Statistics: **mean**, **p10**, **p25**, **median (p50)**, **p75**, **p90**. Plots use **Age** on the x-axis.

Standalone script: `python scripts/run_lifecycle_profiles.py` (writes CSV + figure under `output/`).


In [ ]:
# Larger cohort for smoother quantiles (re-uses return params from §1).
par_lc = HousingFinanceParams(
    T=20,
    age_min=25.0,
    age_max=80.0,
    Nm=14,
    Nh=10,
    Nz=3,
    n_c=10,
    simN=1000,
    sim_seed=1,
    returns=ret_params,
    m_max=6.0,
    h_max=4.0,
)
model_lc = HousingFinanceModel(par_lc)
print(
    f"Age grid ({par_lc.age_min:.0f}-{par_lc.age_max:.0f}), "
    f"Delta={model_lc.period_years:.4f} years/period"
)
display(pd.DataFrame({"t": np.arange(par_lc.T), "age": model_lc.ages}))

print("Solving lifecycle cohort model...")
model_lc.solve()
print("Simulating...")
model_lc.simulate()

profiles = profiles_from_sim(model_lc.sim, variables=DEFAULT_VARIABLES, ages=model_lc.ages)
csv_path = OUT / "lifecycle_cohort_profiles.csv"
profiles.to_csv(csv_path, index=False)
print("Wrote", csv_path)

# Wide tables by variable (index = chronological age)
for var in DEFAULT_VARIABLES:
    print(f"\n--- {var} ---")
    display(
        profiles.loc[profiles["variable"] == var]
        .drop(columns=["variable"])
        .set_index("age")
        .round(3)
    )

ok_profiles = (
    profiles["mean"].notna().all()
    and (profiles.loc[profiles["variable"] == "net_worth", "mean"] > 0).all()
    and (profiles.loc[profiles["variable"] == "c", "mean"] > 0).all()
    and abs(float(profiles["age"].min()) - par_lc.age_min) < 1e-9
    and abs(float(profiles["age"].max()) - par_lc.age_max) < 1e-9
)
checks["lifecycle_profiles"] = bool(ok_profiles)
print("PASS" if ok_profiles else "FAIL", "— lifecycle cohort profiles")


In [ ]:
fig = plot_lifecycle_profiles(profiles, variables=DEFAULT_VARIABLES)
fig_path = OUT / "lifecycle_cohort_profiles.png"
fig.savefig(fig_path, dpi=120)
plt.show()
print("Wrote", fig_path)

## 4. Simulation modes (cohort vs panel; drawn vs provided initials)

Four demos of the calibration-ready simulators (`initial.py`, `simulate.py`).

| Demo | Mode | Initials |
|------|------|----------|
| 4.1 | Cohort lifecycle | Drawn |
| 4.2 | Cohort lifecycle | Provided `InitialSample` |
| 4.3 | Mixed-age panel (~20 years) | Drawn |
| 4.4 | Mixed-age panel (~20 years) | Provided `InitialSample` |

Shared settings: **T = 20** life periods spanning ages 25–80, **N = 1000** agents.
Each cell reports wall-clock time for the simulation step (solve is done once below).


In [ ]:
# Shared solved model for §4 (T≈20 life periods)
N_AGENTS = 1000
T_LIFE = 20

par_sim = HousingFinanceParams(
    T=T_LIFE,
    age_min=25.0,
    age_max=80.0,
    Nm=14,
    Nh=10,
    Nz=3,
    n_c=10,
    simN=N_AGENTS,
    sim_seed=11,
    returns=ret_params,
    m_max=6.0,
    h_max=4.0,
)
model_sim = HousingFinanceModel(par_sim)

# Calendar steps for ~20 years of panel time
PANEL_YEARS = 20.0
K_PANEL = max(1, int(round(PANEL_YEARS / model_sim.period_years)))
calendar_years = np.arange(K_PANEL, dtype=float) * model_sim.period_years

print(
    f"T={par_sim.T}, ages {model_sim.ages[0]:.0f}–{model_sim.ages[-1]:.0f}, "
    f"period_years Δ={model_sim.period_years:.3f}"
)
print(
    f"Panel: K={K_PANEL} steps ≈ {K_PANEL * model_sim.period_years:.1f} years "
    f"(target {PANEL_YEARS:.0f}y), N={N_AGENTS}"
)

t0 = time.perf_counter()
model_sim.solve()
solve_sec = time.perf_counter() - t0
print(f"Solve wall time: {solve_sec:.2f}s  |  V[0] mean = {float(np.mean(model_sim.sol.V[0])):.4f}")


### 4.1 Cohort + drawn initials


In [ ]:
t0 = time.perf_counter()
model_sim.simulate(distribution=InitialDistribution(), seed=21, n=N_AGENTS)
sec = time.perf_counter() - t0
print(f"Cohort + drawn: {sec:.2f}s for N={N_AGENTS}, T={par_sim.T}")

prof_c1 = profiles_from_sim(model_sim.sim, variables=DEFAULT_VARIABLES, ages=model_sim.ages)
fig = plot_lifecycle_profiles(prof_c1, variables=DEFAULT_VARIABLES, xlabel="Age")
fig.suptitle("4.1 Cohort + drawn initials — mean and quantiles", y=1.02)
fig.tight_layout()
fig_path = OUT / "sim_cohort_drawn_profiles.png"
fig.savefig(fig_path, dpi=120, bbox_inches="tight")
plt.show()
print("Wrote", fig_path)

ok = model_sim.sim.m.shape == (T_LIFE, N_AGENTS) and np.isfinite(model_sim.sim.net_worth).all()
checks["sim_cohort_drawn"] = bool(ok)
print("PASS" if ok else "FAIL", "— cohort + drawn")


### 4.2 Cohort + provided `InitialSample`


In [ ]:
# Build an explicit N-agent sample (placeholder "microdata"), then pass it in
rng_prov = np.random.default_rng(22)
init_cohort = InitialDistribution().draw(
    N_AGENTS, rng_prov, Nz=par_sim.Nz, Nr=model_sim.Rf.size, T=par_sim.T
)
# Force young cohort
init_cohort = init_cohort.with_t_age(0)

t0 = time.perf_counter()
model_sim.simulate(initial=init_cohort, seed=22)
sec = time.perf_counter() - t0
print(f"Cohort + provided: {sec:.2f}s for N={N_AGENTS}, T={par_sim.T}")

ok_init = np.allclose(model_sim.sim.m[0], init_cohort.m) and np.allclose(
    model_sim.sim.h[0], init_cohort.h
)
print("Initial m,h respected at t=0:", ok_init)

prof_c2 = profiles_from_sim(model_sim.sim, variables=DEFAULT_VARIABLES, ages=model_sim.ages)
fig = plot_lifecycle_profiles(prof_c2, variables=DEFAULT_VARIABLES, xlabel="Age")
fig.suptitle("4.2 Cohort + provided InitialSample — mean and quantiles", y=1.02)
fig.tight_layout()
fig_path = OUT / "sim_cohort_provided_profiles.png"
fig.savefig(fig_path, dpi=120, bbox_inches="tight")
plt.show()
print("Wrote", fig_path)

checks["sim_cohort_provided"] = bool(ok_init and model_sim.sim.m.shape == (T_LIFE, N_AGENTS))
print("PASS" if checks["sim_cohort_provided"] else "FAIL", "— cohort + provided sample")


### 4.3 Panel + drawn mixed ages (~20 years)

Cross-section of mixed ages stepped for $K$ calendar periods ($K\cdot\Delta\approx 20$ years).
Reports mean and major quantiles of net worth, wealth components, income, consumption, and age.


In [ ]:
t0 = time.perf_counter()
model_sim.simulate_panel(
    K_PANEL,
    distribution=InitialDistribution(mixed_ages=True),
    on_terminal="replace_young",
    seed=31,
    n=N_AGENTS,
)
sec = time.perf_counter() - t0
print(
    f"Panel + drawn: {sec:.2f}s for N={N_AGENTS}, K={K_PANEL} "
    f"(≈{K_PANEL * model_sim.period_years:.1f} years)"
)

P = model_sim.sim.panel
panel_paths = {k: getattr(P, k) for k in PANEL_VARIABLES}
prof_p1 = cohort_lifecycle_profiles(panel_paths, variables=PANEL_VARIABLES, ages=calendar_years)
fig = plot_lifecycle_profiles(
    prof_p1, variables=PANEL_VARIABLES, xlabel="Calendar years since start"
)
fig.suptitle("4.3 Panel + drawn — cross-section mean and quantiles", y=1.02)
fig.tight_layout()
fig_path = OUT / "sim_panel_drawn_profiles.png"
fig.savefig(fig_path, dpi=120, bbox_inches="tight")
plt.show()
print("Wrote", fig_path)

# Age histogram start vs end
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(P.chronological_age[0][P.alive[0]], bins=15, alpha=0.55, label="start")
ax.hist(P.chronological_age[-1][P.alive[-1]], bins=15, alpha=0.55, label="end")
ax.set_xlabel("Age")
ax.set_title("4.3 Age distribution (start vs end)")
ax.legend()
fig.tight_layout()
plt.show()

ok = P.net_worth.shape == (K_PANEL, N_AGENTS) and int(P.alive[-1].sum()) == N_AGENTS
checks["sim_panel_drawn"] = bool(ok)
print("PASS" if ok else "FAIL", "— panel + drawn mixed ages")


### 4.4 Panel + provided mixed-age sample (~20 years)

Explicit `InitialSample` of N agents with mixed `t_age`. Terminal rule: `replace_young`
(also available: `exit`).


In [ ]:
rng_p = np.random.default_rng(41)
init_panel = InitialDistribution(mixed_ages=True).draw(
    N_AGENTS, rng_p, Nz=par_sim.Nz, Nr=model_sim.Rf.size, T=par_sim.T
)

t0 = time.perf_counter()
model_sim.simulate_panel(
    n_periods=K_PANEL,
    initial=init_panel,
    on_terminal="replace_young",
    seed=41,
)
sec = time.perf_counter() - t0
print(
    f"Panel + provided: {sec:.2f}s for N={N_AGENTS}, K={K_PANEL} "
    f"(≈{K_PANEL * model_sim.period_years:.1f} years)"
)

Pp = model_sim.sim.panel
ok_init = np.allclose(Pp.m[0], init_panel.m) and np.array_equal(Pp.t_age[0], init_panel.t_age)
print("Initial m and t_age respected at calendar 0:", ok_init)

panel_paths = {k: getattr(Pp, k) for k in PANEL_VARIABLES}
prof_p2 = cohort_lifecycle_profiles(panel_paths, variables=PANEL_VARIABLES, ages=calendar_years)
fig = plot_lifecycle_profiles(
    prof_p2, variables=PANEL_VARIABLES, xlabel="Calendar years since start"
)
fig.suptitle("4.4 Panel + provided sample — cross-section mean and quantiles", y=1.02)
fig.tight_layout()
fig_path = OUT / "sim_panel_provided_profiles.png"
fig.savefig(fig_path, dpi=120, bbox_inches="tight")
plt.show()
print("Wrote", fig_path)

checks["sim_panel_provided"] = bool(ok_init and Pp.m.shape == (K_PANEL, N_AGENTS))
print("PASS" if checks["sim_panel_provided"] else "FAIL", "— panel + provided sample")


## 5. `DecisionSimulation` callable (pre-SMM primitive)

Configure once (flags, $N$, $K$, initials); call as $f(\theta)$:

**params → solve decision rules → cohort + panel sims → placeholder statistics.**

Uses the same **T ≈ 20**, **N = 1000**, and panel horizon ≈ 20 years as §4.


In [ ]:
par_ds = HousingFinanceParams(
    T=T_LIFE,
    age_min=25.0,
    age_max=80.0,
    Nm=14,
    Nh=10,
    Nz=3,
    n_c=10,
    simN=N_AGENTS,
    returns=ret_params,
    m_max=6.0,
    h_max=4.0,
)
decision_sim = DecisionSimulation(
    params_template=par_ds,
    run_cohort=True,
    run_panel=True,
    simN=N_AGENTS,
    panel_n_periods=K_PANEL,
    on_terminal="replace_young",
    seed=7,
    keep_model=False,
)

t0 = time.perf_counter()
res0 = decision_sim({})
sec0 = time.perf_counter() - t0
print(f"DecisionSimulation baseline: {sec0:.2f}s (solve + cohort + panel)")

t0 = time.perf_counter()
res1 = decision_sim({"returns_sigma_f": 0.28})
sec1 = time.perf_counter() - t0
print(f"DecisionSimulation high sigma_f: {sec1:.2f}s")

stats_cmp = pd.DataFrame({"baseline": res0.statistics, "high_sigma_f": res1.statistics})
display(stats_cmp)

ok_ds = (
    res0.cohort is not None
    and res0.panel is not None
    and len(res0.statistics) > 0
    and np.isfinite(res0.moment_vector).all()
    and not np.allclose(res0.moment_vector, res1.moment_vector)
)
checks["decision_simulation"] = bool(ok_ds)
print("PASS" if ok_ds else "FAIL", "— DecisionSimulation callable")
print("moment_names:", res0.moment_names)


## 6. OLG inequality panel (low vs high financial vol)

Uses coarse grids for speed. Compares net-worth Gini paths.


In [ ]:
def run_olg(sigma_f: float, calendar_t: int = 24, agents_per_cohort: int = 80, seed: int = 1) -> pd.DataFrame:
    ret = CanadaReturnParams(n_f=3, n_h=3, sigma_f=sigma_f, sigma_h=0.10, corr=0.35)
    p = HousingFinanceParams(
        T=6, Nm=12, Nh=8, Nz=3, n_c=8,
        simN=agents_per_cohort, sim_seed=seed,
        returns=ret, m_max=6.0, h_max=4.0,
    )
    m = HousingFinanceModel(p)
    m.solve()

    n_cohorts = calendar_t
    wealth = np.full((calendar_t, n_cohorts * agents_per_cohort), np.nan)
    alive = np.zeros_like(wealth, dtype=bool)

    for k in range(n_cohorts):
        m.par.sim_seed = seed + k
        m.par.simN = agents_per_cohort
        m.simulate()
        nw = m.sim.net_worth
        col0 = k * agents_per_cohort
        col1 = col0 + agents_per_cohort
        for age in range(nw.shape[0]):
            t = k + age
            if t >= calendar_t:
                break
            wealth[t, col0:col1] = nw[age]
            alive[t, col0:col1] = True

    return wealth_inequality_path(wealth, alive_mask=alive)

print("OLG low vol...")
path_lo = run_olg(0.10)
print("OLG high vol...")
path_hi = run_olg(0.22)

path_lo.to_csv(OUT / "hf_olg_gini_lowvol.csv", index=False)
path_hi.to_csv(OUT / "hf_olg_gini_highvol.csv", index=False)

print("Low vol:", summarize_path(path_lo))
print("High vol:", summarize_path(path_hi))

ok_olg = (
    np.isfinite(path_lo["gini"].iloc[-1])
    and np.isfinite(path_hi["gini"].iloc[-1])
    and path_lo["n_alive"].iloc[-1] > 0
)
checks["olg_inequality"] = bool(ok_olg)
print("PASS" if ok_olg else "FAIL", "— OLG inequality panel")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(path_lo["t"], path_lo["gini"], label="sigma_f=0.10")
ax.plot(path_hi["t"], path_hi["gini"], label="sigma_f=0.22")
ax.set_xlabel("calendar time")
ax.set_ylabel("net-worth Gini")
ax.set_title("Housing-finance OLG: inequality transition (PE)")
ax.legend()
fig.tight_layout()
fig_path = OUT / "hf_olg_gini_transition.png"
fig.savefig(fig_path, dpi=120)
plt.show()
print("Wrote", fig_path)

## 7. Wealth-dependent financial volatility (simulation layer)

Solve with constant sigma_f; compare sims with `wealth_vol_kappa=0` vs `>0`.


In [ ]:
ret_w = CanadaReturnParams(n_f=3, n_h=3, sigma_f=0.14, sigma_h=0.10, corr=0.35)
par_w = HousingFinanceParams(
    T=6, Nm=12, Nh=8, Nz=3, n_c=8, simN=2000,
    returns=ret_w, m_max=6.0, h_max=4.0,
)
model_w = HousingFinanceModel(par_w)
print("Solving (constant vol at solve time)...")
model_w.solve()

model_w.par.sim_seed = 11
model_w.simulate(wealth_vol_kappa=0.0)
nw_const = model_w.sim.net_worth.copy()
gini_const = [gini(nw_const[t]) for t in range(nw_const.shape[0])]

model_w.par.sim_seed = 11
model_w.simulate(wealth_vol_kappa=0.8, wealth_vol_norm=4.0)
nw_scaled = model_w.sim.net_worth.copy()
gini_scaled = [gini(nw_scaled[t]) for t in range(nw_scaled.shape[0])]

scaled_chain = scale_financial_vol(discretize_canada_returns(ret_w), vol_mult=1.5)
print("scale_financial_vol(1.5):", scaled_chain.summary())

ok_vol = np.isfinite(gini_const[-1]) and np.isfinite(gini_scaled[-1])
checks["wealth_vol"] = bool(ok_vol)
print(f"Final Gini const={gini_const[-1]:.3f}, wealth-scaled={gini_scaled[-1]:.3f}")
print("PASS" if ok_vol else "FAIL", "— wealth-dependent vol")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(nw_const.mean(axis=1), label="const vol")
axes[0].plot(nw_scaled.mean(axis=1), label="wealth-scaled vol")
axes[0].set_title("Mean net worth")
axes[0].legend()
axes[1].plot(gini_const, label="const vol")
axes[1].plot(gini_scaled, label="wealth-scaled vol")
axes[1].set_title("Wealth Gini")
axes[1].legend()
fig.suptitle("Wealth-dependent financial volatility (sim approximation)")
fig.tight_layout()
fig_path = OUT / "wealth_dependent_vol.png"
fig.savefig(fig_path, dpi=120)
plt.show()

np.savetxt(
    OUT / "wealth_vol_gini_compare.csv",
    np.column_stack([gini_const, gini_scaled]),
    delimiter=",",
    header="gini_const,gini_wealth_scaled",
    comments="",
)
print("Wrote", fig_path)

## 8. Age-income profile and De Nardi bequest checks

- Default g(age) = 0 so y_age_factor is all ones.
- With equest_phi>0, terminal consumption is below liquidated resources X = m+(1-tau)h on average.


In [ ]:
# (i) default age profile is flat in logs
ok_g0 = np.allclose(model.y_age_factor, 1.0) and np.allclose(model.g_age, 0.0)
print("y_age_factor:", model.y_age_factor[:5], "..." if model.y_age_factor.size > 5 else "")
checks["age_income_g0"] = bool(ok_g0)
print("PASS" if ok_g0 else "FAIL", "— g(age)=0 => y_age_factor=1")

# (ii) bequest: compare terminal c/X for phi=0 vs phi>0 (small grids)
par_b0 = HousingFinanceParams(
    T=4, age_min=25.0, age_max=80.0,
    Nm=8, Nh=6, Nz=3, n_c=12, simN=20,
    returns=ret_params, m_max=4.0, h_max=2.0,
    bequest_phi=0.0,
)
par_b1 = HousingFinanceParams(
    T=4, age_min=25.0, age_max=80.0,
    Nm=8, Nh=6, Nz=3, n_c=12, simN=20,
    returns=ret_params, m_max=4.0, h_max=2.0,
    bequest_phi=2.0, bequest_k=0.1,
)
mb0 = HousingFinanceModel(par_b0)
mb1 = HousingFinanceModel(par_b1)
mb0.solve()
mb1.solve()
X = mb0.grid_m[:, None] + (1.0 - par_b0.tau) * mb0.grid_h[None, :]
mask = X > 1e-8
c0 = mb0.sol.c[-1, :, :, 0, 0]
c1 = mb1.sol.c[-1, :, :, 0, 0]
ratio0 = float(np.mean(c0[mask] / X[mask]))
ratio1 = float(np.mean(c1[mask] / X[mask]))
print(f"terminal mean c/X: phi=0 -> {ratio0:.3f}; phi=2 -> {ratio1:.3f}")

ok_beq = abs(ratio0 - 1.0) < 1e-6 and ratio1 < ratio0 - 1e-3
checks["bequest_terminal"] = bool(ok_beq)
print("PASS" if ok_beq else "FAIL", "— De Nardi bequest lowers terminal c/X")


## 9. Summary checklist


In [ ]:
summary_df = pd.DataFrame(
    [{"check": k, "pass": v} for k, v in checks.items()]
)
display(summary_df)

all_ok = all(checks.values()) and len(checks) == 4
print("\n" + ("ALL CHECKS PASSED" if all_ok else "SOME CHECKS FAILED"))
print("Outputs in:", OUT)
print("Expected files:")
for name in [
    "housing_finance_baseline.png",
    "hf_olg_gini_lowvol.csv",
    "hf_olg_gini_highvol.csv",
    "hf_olg_gini_transition.png",
    "wealth_dependent_vol.png",
    "wealth_vol_gini_compare.csv",
]:
    p = OUT / name
    print(" ", "OK" if p.exists() else "MISSING", p.name)